# Gráfico de anomalías de rodilla

Quería ver qué etiquetas aparecían más entre los estudios que tenían las 12 completas. El notebook anterior guardó el gráfico y mostró 58 estudios completos. De esos, 8 tenían una sola etiqueta positiva.

Si tengo el `train.csv`, lo dejo en `datos/crudos/rsna/train.csv` y puedo volver a contar todo. Si falta el archivo, uso los conteos que se leen en el gráfico anterior para dibujarlo de nuevo. En ese caso los resultados son del gráfico guardado.

[Competencia y archivo train.csv](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/data)


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# 1. Encontrar la carpeta del repo y el CSV.
carpetas = [Path.cwd(), *Path.cwd().parents]
raiz = next((p for p in carpetas if (p / "funciones/datos.py").is_file()), None)
if raiz is None:
    raise FileNotFoundError("Abrí el notebook desde la carpeta del repositorio.")
ruta = raiz / "datos/crudos/rsna/train.csv"

etiquetas = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]

if ruta.is_file():
    # 2. Leer el ID y las etiquetas. Para el gráfico no necesito los informes.
    train = pd.read_csv(ruta, usecols=["StudyInstanceUID", *etiquetas])
    if train["StudyInstanceUID"].isna().any() or not train["StudyInstanceUID"].is_unique:
        raise ValueError("Tiene que haber un ID distinto por estudio.")
    valores = train[etiquetas].apply(pd.to_numeric, errors="raise")
    if not (valores.isin([0, 1]) | valores.isna()).all().all():
        raise ValueError("Las etiquetas tienen que ser 0, 1 o NA.")
    completas = valores.dropna()
    cantidad_estudios = len(completas)
    if cantidad_estudios == 0:
        raise ValueError("El CSV no tiene estudios con las 12 etiquetas completas.")
    cantidad_unicas = int(completas.sum(axis=1).eq(1).sum())
    conteo = completas.sum().astype(int).sort_values(ascending=False)
    fuente = "Cálculo con train.csv"
else:
    # Conteos leídos del gráfico guardado en el notebook original.
    cantidad_estudios = 58
    cantidad_unicas = 8
    conteo = pd.Series({
        "Effusion": 35, "Synovitis": 27, "Medial Meniscus": 26, "ACL": 24,
        "Lateral Meniscus": 23, "PF OA": 21, "Contusion": 19, "Fracture": 18,
        "Medial OA": 15, "Baker's": 12, "Lateral OA": 11, "MCL": 9,
    }, dtype=int)
    fuente = "Conteos recuperados del gráfico anterior"

# 3. Ver que cada barra esté entre 0 y la cantidad de estudios.
assert set(conteo.index) == set(etiquetas)
assert conteo.between(0, cantidad_estudios).all()
print(fuente)
print("Estudios con todas las etiquetas:", cantidad_estudios)
print("Estudios con una sola etiqueta positiva:", cantidad_unicas)
print(conteo.to_string())


In [ ]:
# 4. Graficar la cantidad de positivos por etiqueta.
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(conteo.index, conteo.values)
ax.set(
    title=f"Etiquetas positivas en {cantidad_estudios} estudios completos",
    xlabel="Anomalía",
    ylabel="Cantidad de estudios",
)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
fig.text(0.01, 0.01, fuente, fontsize=9)
fig.tight_layout(rect=(0, 0.04, 1, 1))
salida = raiz / "salidas"
salida.mkdir(exist_ok=True)
fig.savefig(salida / "rsna.png", dpi=140)
plt.show()


## Qué se ve en el gráfico

En el gráfico guardado, Effusion quedó primera con 35 positivos y MCL última con 9. Un estudio puede entrar en más de una barra, así que sumar las barras cuenta etiquetas positivas. El gráfico usa los estudios con las 12 etiquetas completas. Para hablar de toda la base tendría que ver cómo se eligieron esos estudios y qué pasa con los que quedaron afuera.

[Notebook anterior con el gráfico guardado](https://github.com/ETHARTIC/Class-activities/blob/a4ea2f724513a3832f17a5c72efc6c808c7299b6/Actividades-de-clase/scripts/trabajo-creacion-grafico.ipynb)
